In [8]:
import requests
import pandas as pd
import uuid

# Basis configuratie gebaseerd op de technische documentatie
BASE_URL = "https://api.ah.nl"
STORE_ID = "1558"  # Dit is het unieke ID voor AH Woenselse Markt Eindhoven

# Verplichte headers om de AH app na te bootsen
headers = {
    "User-Agent": "Appie/9.28 (iPhone17,3; iPhone; CPU OS 26_1 like Mac OS X)",
    "x-application": "AHWEBSHOP",
    "x-clientname": "appie-ios",
    "x-": "9.28",
    "x-fraud-detection-installation-id": str(uuid.uuid4()), # Een unieke ID per sessie
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [9]:
def get_anonymous_token():
    auth_url = f"{BASE_URL}/mobile-auth/v1/auth/token/anonymous"
    payload = {"clientId": "appie-ios"}
    
    response = requests.post(auth_url, json=payload, headers=headers)
    response.raise_for_status() # Geeft een foutmelding als het misgaat
    
    token_data = response.json()
    return token_data['access_token']

# Activeer de sleutel voor alle volgende verzoeken
access_token = get_anonymous_token()
headers["Authorization"] = f"Bearer {access_token}"
print("Handshake succesvol: Token opgehaald.")

Handshake succesvol: Token opgehaald.


In [10]:
bargain_query = """
query GetBargains($storeId: String!) {
  bargainItems(storeId: $storeId) {
    categoryTitle  # <--- Deze voegt de categorie (zoals Vlees) toe
    product {
      title
      brand
      salesUnitSize
    }
    bargainPrice {
      priceWas
      priceNow
    }
    markdown {
      markdownPercentage
      markdownExpirationDate
    }
    stock
  }
}
"""

def fetch_laatste_kans(store_id):
    url = f"{BASE_URL}/graphql"
    
    graphql_headers = headers.copy()
    graphql_headers.update({
        "x-apollo-operation-name": "GetBargains",
        "x-apollo-operation-type": "query",
        "apollographql-client-name": "nl.ah.Appie-apollo-ios",
        "apollographql-client-version": "9.28-260102201630"
    })
    
    payload = {
        'query': bargain_query, 
        'variables': {'storeId': store_id},
        'operationName': 'GetBargains'
    }
    
    response = requests.post(url, json=payload, headers=graphql_headers)
    data = response.json()
    
    if 'errors' in data:
        print("❌ GraphQL Foutmelding gevonden:")
        for error in data['errors']:
            print(f" - {error.get('message')}")
        return None
        
    return data.get('data', {}).get('bargainItems')

# Haal de ruwe data opnieuw op
raw_items = fetch_laatste_kans(STORE_ID)

if raw_items:
    print(f"✅ Succes! {len(raw_items)} producten gevonden op de Woenselse Markt.")
else:
    print("❌ Geen data ontvangen. Controleer de output hierboven.")

✅ Succes! 152 producten gevonden op de Woenselse Markt.


In [11]:
# Gebruik json_normalize om geneste velden (zoals product.title) plat te slaan
df = pd.json_normalize(raw_items)

# Optioneel: Kolomnamen opschonen voor gemak
df.columns = [c.replace('product.', '').replace('bargainPrice.', '').replace('markdown.', '') for c in df.columns]

# Sorteer op de hoogste korting
df = df.sort_values(by='markdownPercentage', ascending=False)

# Toon de live status
display(df.head(10))

,categoryTitle,stock,priceWas,priceNow,markdownPercentage,markdownExpirationDate,title,brand,salesUnitSize
92,"Zuivel, eieren",4,2.49,0.75,70,2026-01-26,Optimel Magere vla vanille,Optimel,1 l
101,Bakkerij,1,3.29,0.99,70,2026-01-28,Dr. Oetker Kwarktaart eigen fruit bakmix,Dr. Oetker,210 g
82,Vleeswaren,3,1.99,0.60,70,2026-01-29,AH Duitse theeworst,AH,125 g
123,"Koek, snoep, chocolade",2,2.99,1.50,50,2026-01-25,Fruittella Berries & cherry,Fruittella,200 g
121,"Koek, snoep, chocolade",6,1.99,1.00,50,2026-01-25,Red Band Pretletters zoet,Red Band,125 g
122,"Koek, snoep, chocolade",4,2.79,1.40,50,2026-01-25,Mentos Gum Sour strawberry,Mentos Gum,56 g
125,"Koek, snoep, chocolade",1,1.99,1.00,50,2026-01-25,Food2Smile Very berry,Food2Smile,90 g
124,"Koek, snoep, chocolade",1,3.99,2.00,50,2026-01-25,Stimorol Spearmint,Stimorol,101.5 g
136,"Koek, snoep, chocolade",11,2.29,1.15,50,2026-02-01,Galler Wit kokosnoot,Galler,70 g
135,"Koek, snoep, chocolade",14,4.19,2.10,50,2026-02-01,Lotus Biscoff Speculoos witte chocolade stukjes,Lotus Biscoff,180 g


In [14]:
import streamlit as st
import pandas as pd
import requests
import uuid
import os

# --- PAGINA CONFIGURATIE ---
st.set_page_config(page_title="Appie Koopjes Sniper", page_icon="🛒", layout="wide")

# --- AH API HELPERS ---
def get_ah_token():
    headers = {"User-Agent": "Appie/9.28", "x-application": "AHWEBSHOP"}
    res = requests.post("https://api.ah.nl/mobile-auth/v1/auth/token/anonymous", 
                         json={"clientId": "appie-ios"}, headers=headers)
    return res.json().get('access_token')

def get_live_bargains(store_id):
    token = get_ah_token()
    headers = {"Authorization": f"Bearer {token}", "x-application": "AHWEBSHOP"}
    query = """
    query GetBargains($storeId: String!) {
      bargainItems(storeId: $storeId) {
        categoryTitle
        product { title brand images { url width } }
        bargainPrice { priceNow priceWas }
        markdown { markdownPercentage }
      }
    }
    """
    res = requests.post("https://api.ah.nl/graphql", 
                         json={'query': query, 'variables': {'storeId': str(store_id)}}, 
                         headers=headers)
    return res.json().get('data', {}).get('bargainItems', [])

# --- UI ONTWERP ---
st.title("🛒 Appie Koopjes Sniper")
st.markdown("Ontdek de beste 'Laatste Kans' deals van jouw Albert Heijn.")

# Sidebar: Store Selector
with st.sidebar:
    st.header("Instellingen")
    # Voor nu een paar bekende, maar je kunt dit uitbreiden
    stores = {"Woenselse Markt (1558)": "1558", "Eindhoven XL (1177)": "1177", "Amsterdam Damrak (1166)": "1166"}
    selected_store_name = st.selectbox("Kies een winkel:", list(stores.keys()))
    STORE_ID = stores[selected_store_name]
    
    st.info("De robot op GitHub verzamelt momenteel alleen data voor de Woenselse Markt.")

# Live Data ophalen
items = get_live_bargains(STORE_ID)

if items:
    df = pd.json_normalize(items)
    
    # 70% SECTIE
    st.subheader("🔥 70% Korting Knallers")
    snipers = df[df['markdown.markdownPercentage'] == 70]
    
    if not snipers.empty:
        # Maak kolommen voor de 'kaarten'
        cols = st.columns(4)
        for i, (idx, row) in enumerate(snipers.iterrows()):
            with cols[i % 4]:
                # Haal het plaatje op (meestal de eerste in de lijst)
                img_url = row['product.images'][0]['url'] if row['product.images'] else ""
                st.image(img_url, use_container_width=True)
                st.write(f"**{row['product.title']}**")
                st.write(f"~~€{row['bargainPrice.priceWas']}~~ → **€{row['bargainPrice.priceNow']}**")
                st.caption(f"Categorie: {row['categoryTitle']}")
    else:
        st.write("Geen 70% deals op dit moment.")

    st.divider()

    # OVERZICHTSTABEL
    st.subheader("📦 Alle huidige aanbiedingen")
    # Opschonen voor de tabel
    display_df = df[['categoryTitle', 'product.title', 'bargainPrice.priceWas', 'bargainPrice.priceNow', 'markdown.markdownPercentage']]
    st.dataframe(display_df, use_container_width=True)

else:
    st.error("Geen data gevonden voor deze winkel. De bak is waarschijnlijk leeg!")

# Historie Sectie (Data Science!)
if os.path.exists("koopjes_historie.csv"):
    st.divider()
    st.subheader("📈 Historische Patronen (Data Science)")
    hist_df = pd.read_csv("koopjes_historie.csv")
    st.line_chart(hist_df.groupby('timestamp').size())
    st.write("Aantal gevonden koopjes over de tijd.")



2026-01-25 13:58:21.444 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 13:58:21.446 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 13:58:21.593 
  command:

    streamlit run c:\Users\20193623\Desktop\Appie App\venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-01-25 13:58:21.594 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 13:58:21.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 13:58:21.596 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 13:58:21.597 Thread 'MainThread': missing ScriptRunContext! This warning can be i